In [1]:
!pip install -qU langchain-google-genai pydantic

In [4]:
import os
import json
from typing import List, Optional
from pydantic import BaseModel, Field
from langchain_google_genai import ChatGoogleGenerativeAI
from google.colab import userdata

# ------------------------------------------------------------------------------
# 1. DATA SCHEMA (The "Contract" between AI and Backend)
# ------------------------------------------------------------------------------
class FeedbackItem(BaseModel):
    error_type: str = Field(description="Must be: grammar, vocabulary, or pronunciation")
    error_phrase: str = Field(description="The phrase containing the error (in English)")
    correction: str = Field(description="The academic improvement (in English)")
    reasoning: str = Field(description="Explanation in VIETNAMESE for local students.")

class Scores(BaseModel):
    overall: float
    grammar: float
    vocabulary: float
    fluency: float
    pronunciation: float

class IELTSResult(BaseModel):
    transcript: str
    scores: Scores
    feedback: List[FeedbackItem]

# ------------------------------------------------------------------------------
# 2. CORE ENGINE CLASS
# ------------------------------------------------------------------------------
class IELTSEngine:
    def __init__(self):
        # Fetch API key securely from Colab Secrets
        try:
            self.api_key = userdata.get('GEMINI_API_KEY')
            os.environ["GOOGLE_API_KEY"] = self.api_key
        except Exception:
            raise ValueError("SECRET NOT FOUND: Please add GEMINI_API_KEY to Colab Secrets.")

        # Initialize Model (Using 1.5-Flash for maximum stability)
        self.llm = ChatGoogleGenerativeAI(model="gemini-flash-latest", temperature=0)
        self.structured_llm = self.llm.with_structured_output(IELTSResult)

    def evaluate(self, text: str) -> dict:
        prompt = f"""
        You are an expert IELTS examiner. Evaluate this transcript: "{text}".
        1. Rate based on IELTS rubrics.
        2. Explain errors clearly in VIETNAMESE (reasoning field).
        3. Keep corrections in English.
        """
        try:
            response = self.structured_llm.invoke(prompt)
            return response.model_dump()
        except Exception as e:
            return {"error": "API Error", "details": str(e)}

# ------------------------------------------------------------------------------
# 3. EXECUTION
# ------------------------------------------------------------------------------
if __name__ == "__main__":
    engine = IELTSEngine()
    test_text = "I study Data Analysis. It requires a lot of logical thinking."

    print("🤖 AI is grading... please wait...")
    result = engine.evaluate(test_text)

    print("\n" + "="*50)
    print("✅ FINAL STRUCTURED JSON OUTPUT")
    print("="*50)
    print(json.dumps(result, indent=2, ensure_ascii=False))

🤖 AI is grading... please wait...

✅ FINAL STRUCTURED JSON OUTPUT
{
  "transcript": "I study Data Analysis. It requires a lot of logical thinking.",
  "scores": {
    "overall": 6.0,
    "grammar": 6.0,
    "vocabulary": 6.0,
    "fluency": 6.0,
    "pronunciation": 6.0
  },
  "feedback": [
    {
      "error_type": "vocabulary",
      "error_phrase": "a lot of",
      "correction": "a significant amount of",
      "reasoning": "Cụm từ 'a lot of' khá thông dụng trong giao tiếp hàng ngày nhưng thiếu tính học thuật. Sử dụng 'a significant amount of' hoặc 'extensive' sẽ giúp nâng cao điểm từ vựng."
    },
    {
      "error_type": "grammar",
      "error_phrase": "I study Data Analysis. It requires",
      "correction": "I study Data Analysis, which requires",
      "reasoning": "Việc sử dụng hai câu đơn ngắn làm cho bài nói bị ngắt quãng. Sử dụng mệnh đề quan hệ 'which' để nối hai câu sẽ tạo ra một câu phức, giúp cải thiện điểm ngữ pháp."
    }
  ]
}
